In [1]:
import pandas as pd
import geopandas as gpd

BASE = '/Users/abilindemann/projects/sa-housing-food-insecurity/data/processed/'
RAW  = '/Users/abilindemann/projects/sa-housing-food-insecurity/data/raw/'

# Load Texas block group shapefile and filter to Bexar County
tx = gpd.read_file(RAW + 'bexar_block_groups/tl_2023_48_bg.shp')
bexar = tx[tx['COUNTYFP'] == '029'].copy()

print(f"Texas block groups: {len(tx)}")
print(f"Bexar County block groups: {len(bexar)}")
print(f"CRS: {bexar.crs}")
print(bexar[['GEOID', 'geometry']].head(3))

Texas block groups: 18638
Bexar County block groups: 1139
CRS: EPSG:4269
             GEOID                                           geometry
807   480291820031  POLYGON ((-98.65262 29.59343, -98.65256 29.594...
1139  480291216063  POLYGON ((-98.3413 29.55396, -98.34046 29.5543...
1140  480291216042  POLYGON ((-98.32862 29.53726, -98.32856 29.537...


In [3]:
# Load our overlap scores
scores = pd.read_csv(BASE + 'overlap_scores.csv', dtype={'GEOID': str})

# Merge scores onto shapefile geometries
gdf = bexar.merge(scores, on='GEOID', how='left')

# Convert to WGS84 for web mapping
gdf = gdf.to_crs(epsg=4326)

print(f"Merged rows: {len(gdf)}")
print(f"Nulls in housing_score: {gdf['housing_score'].isnull().sum()}")
print(f"Nulls in quadrant: {gdf['quadrant'].isnull().sum()}")
print(f"CRS: {gdf.crs}")

Merged rows: 1139
Nulls in housing_score: 31
Nulls in quadrant: 0
CRS: EPSG:4326


In [5]:
import os

SITE_DATA = '/Users/abilindemann/projects/sa-housing-food-insecurity/site/data/'
os.makedirs(SITE_DATA, exist_ok=True)

# Round floats to keep file size reasonable
float_cols = ['housing_score', 'nearest_grocery_miles', 'food_distance_norm',
              'nearest_food_bank_miles', 'nearest_wic_miles',
              'poverty_rate', 'pct_hispanic', 'lat', 'lon']
for col in float_cols:
    if col in gdf.columns:
        gdf[col] = gdf[col].round(4)

# Export full layer — all block groups with all scores
gdf.to_file(SITE_DATA + 'block_groups.geojson', driver='GeoJSON')

import os
size = os.path.getsize(SITE_DATA + 'block_groups.geojson') / 1024 / 1024
print(f"Saved block_groups.geojson — {size:.2f} MB")

Saved block_groups.geojson — 4.31 MB


In [7]:
# Simplify geometries — tolerance in degrees (~50 meters at SA's latitude)
gdf_simple = gdf.copy()
gdf_simple['geometry'] = gdf_simple['geometry'].simplify(tolerance=0.0005, preserve_topology=True)

gdf_simple.to_file(SITE_DATA + 'block_groups.geojson', driver='GeoJSON')

size = os.path.getsize(SITE_DATA + 'block_groups.geojson') / 1024 / 1024
print(f"Saved simplified block_groups.geojson — {size:.2f} MB")

# Spot check — make sure we didn't lose any rows or break geometries
print(f"Rows: {len(gdf_simple)}")
print(f"Invalid geometries: {(~gdf_simple.geometry.is_valid).sum()}")

Saved simplified block_groups.geojson — 1.41 MB
Rows: 1139
Invalid geometries: 0


In [9]:
gdf_simple['geometry'] = gdf_simple['geometry'].simplify(tolerance=0.001, preserve_topology=True)

gdf_simple.to_file(SITE_DATA + 'block_groups.geojson', driver='GeoJSON')

size = os.path.getsize(SITE_DATA + 'block_groups.geojson') / 1024 / 1024
print(f"Saved block_groups.geojson — {size:.2f} MB")
print(f"Invalid geometries: {(~gdf_simple.geometry.is_valid).sum()}")

Saved block_groups.geojson — 1.32 MB
Invalid geometries: 0


In [11]:
RAW_PROCESSED = BASE

# Grocery stores
grocery = pd.read_csv(RAW_PROCESSED + 'grocery_stores_bexar.csv')
grocery_gdf = gpd.GeoDataFrame(
    grocery,
    geometry=gpd.points_from_xy(grocery['Longitude'], grocery['Latitude']),
    crs='EPSG:4326'
)
grocery_gdf.to_file(SITE_DATA + 'grocery_stores.geojson', driver='GeoJSON')
print(f"Saved grocery_stores.geojson — {len(grocery_gdf)} stores")

# Food banks
food_banks = pd.read_csv(RAW_PROCESSED + 'food_banks_bexar.csv')
fb_gdf = gpd.GeoDataFrame(
    food_banks,
    geometry=gpd.points_from_xy(food_banks['longitude'], food_banks['latitude']),
    crs='EPSG:4326'
)
fb_gdf.to_file(SITE_DATA + 'food_banks.geojson', driver='GeoJSON')
print(f"Saved food_banks.geojson — {len(fb_gdf)} locations")

# WIC clinics
wic = pd.read_csv(RAW_PROCESSED + 'wic_clinics_bexar.csv')
wic_gdf = gpd.GeoDataFrame(
    wic,
    geometry=gpd.points_from_xy(wic['longitude'], wic['latitude']),
    crs='EPSG:4326'
)
wic_gdf.to_file(SITE_DATA + 'wic_clinics.geojson', driver='GeoJSON')
print(f"Saved wic_clinics.geojson — {len(wic_gdf)} clinics")

Saved grocery_stores.geojson — 85 stores
Saved food_banks.geojson — 10 locations
Saved wic_clinics.geojson — 15 clinics


In [13]:
import os

files = os.listdir(SITE_DATA)
for f in sorted(files):
    size = os.path.getsize(SITE_DATA + f) / 1024
    print(f"{f:<30} {size:>8.1f} KB")

block_groups.geojson             1356.3 KB
food_banks.geojson                  2.6 KB
grocery_stores.geojson             29.0 KB
wic_clinics.geojson                 3.7 KB


## Notebook 04 — Export GeoJSON

**Output files in `site/data/`:**
- `block_groups.geojson` — 1,139 Bexar County block groups with all scores (1.36 MB)
- `grocery_stores.geojson` — 85 SNAP-authorized grocery stores (point layer)
- `food_banks.geojson` — 10 food bank / pantry locations (point layer)
- `wic_clinics.geojson` — 15 WIC clinic locations (point layer)

**Method:** Census TIGER/Line 2023 block group shapefile filtered to Bexar County,
merged with overlap scores, geometries simplified (tolerance=0.001°, ~100m),
converted from EPSG:4269 to EPSG:4326 for web mapping.